# Prepare YOLO Dataset
Input:
- Directory containing HLW camera trap MP4 videos
- COCO annotations with frame-video mapping

Output:
- Stratified Train/val/test dataset in YOLO format

In [1]:
import os
import sys
import json

from pathlib import Path
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path("../utilities").resolve()))

from video_dataset_prep_tools import extract_single_video, compute_split_statistics, stratified_video_split, split_report, extract_frames_by_split
from annotation_converter import AnnotationConverter

In [2]:
video_dir = Path("../data/h23/videos")  # Path containing raw camera trap videos
data_dir = Path("../data/h23")  # Path where YOLO dataset will be created

video_arr = [f for f in os.listdir(video_dir) if f.lower().endswith('.mp4')]

## Build Subset COCO JSON from Extracted Frames

In [3]:
images_dir = data_dir / "images"

with open(data_dir / "instances_subset.json") as f:
    coco = json.load(f)

extracted = {p.name for p in images_dir.glob("*.png")}

kept_images = [img for img in coco["images"] if img["file_name"] in extracted]
kept_ids    = {img["id"] for img in kept_images}
kept_anns   = [ann for ann in coco["annotations"] if ann["image_id"] in kept_ids]

subset = {
    "info":        coco.get("info", {}),
    "licenses":    coco.get("licenses", []),
    "categories":  coco["categories"],
    "images":      kept_images,
    "annotations": kept_anns,
}

out_path = data_dir / "instances.json"
with open(out_path, "w") as f:
    json.dump(subset, f)

print(f"Total images in COCO:  {len(coco['images'])}")
print(f"Extracted on disk:     {len(extracted)}")
print(f"Kept images:           {len(kept_images)}")
print(f"Kept annotations:      {len(kept_anns)}")
print(f"Saved → {out_path}")

Total images in COCO:  4843
Extracted on disk:     303
Kept images:           303
Kept annotations:      120
Saved → ../data/h23/instances.json


## Compute Video Statistics

In [4]:
video_stats_df = compute_split_statistics(coco_json_path=str(data_dir / "instances.json"))
display(video_stats_df)

,video,n_total_frames,n_bird_frames,prevalence,stratum
0,IMG_0019,303,120,0.39604,NaN


## Split Dataset

In [5]:
train_videos, val_videos, test_videos = stratified_video_split(video_stats_df, save_dir=data_dir)

split_report(video_stats_df, train_videos, "Train")
split_report(video_stats_df, val_videos,   "Val")
split_report(video_stats_df, test_videos,  "Test")

Split saved to ../data/h23/split.json
Train  |   0 videos |      0 frames | bird prevalence: nan ± nan
Val    |   0 videos |      0 frames | bird prevalence: nan ± nan
Test   |   0 videos |      0 frames | bird prevalence: nan ± nan


## Extract Frames into YOLO Image Directories

In [ ]:
out_dir = data_dir / "images"
split_json = data_dir / "split.json"
    
extract_frames_by_split(split_json, video_dir, out_dir)

KeyboardInterrupt: 

## Convert COCO annotations to YOLO format

In [8]:
def coco_to_yolo_split(train_videos, val_videos, test_videos):
    converter = AnnotationConverter(class_mapping={'bird': 0})
    coco_json_path = "../data/h23/instances_merged.json"

    splits = {
        "train": train_videos,
        "val":   val_videos,
        "test":  test_videos,
    }

    for split_name, videos in splits.items():
        output_dir = data_dir / "labels" / split_name
        converter.coco_to_yolo(
            coco_json_path=coco_json_path,
            output_dir=output_dir,
            use_filename=True,
            video_filter=videos,
        )
        print(f"[{split_name:5s}] labels → {output_dir}")

    converter.create_yaml_config(
    output_dir=data_dir,
    dataset_path='data/h23',
    train_dir='images/train',
    val_dir='images/val',
    test_dir='images/test'
)

coco_to_yolo_split(train_videos, val_videos, test_videos)

Converted 7944 images from COCO to YOLO format
Output directory: ../data/h23/labels/train
[train] labels → ../data/h23/labels/train
Converted 1904 images from COCO to YOLO format
Output directory: ../data/h23/labels/val
[val  ] labels → ../data/h23/labels/val
Converted 1469 images from COCO to YOLO format
Output directory: ../data/h23/labels/test
[test ] labels → ../data/h23/labels/test
Created YOLO config: ../data/h23/yolo.yaml
Classes: {'bird': 0, 'Bird': 0}
